<a href="https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import json, os

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [5]:
from huggingface_hub import whoami

print(whoami(token=HF_TOKEN))

{'type': 'user', 'id': '6a6204151c116dd8e71bc3a2', 'name': 'shami776', 'fullname': 'Muhammad Ehtisham', 'email': 'sharim9185@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/jBaK4KtqwvcI6IaJ-FtEN.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Internship', 'role': 'read', 'createdAt': '2026-08-07T06:30:18.505Z'}}}


In [6]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

info = api.dataset_info("FlyRank/internship-warehouse")

print(info.id)

FlyRank/internship-warehouse


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** *A page is worth reviewing for a CTR fix if it's visible enough to
trust its numbers, and its CTR sits below what other pages at the same position tier typically get
— scored by how far below, times how much traffic is actually at stake.*

**Reason code (one):** `ctr_below_tier_peers` — every flagged row gets this same code; there's only
one rule here, so there's only one reason a row is on the list.

**Action label (one):** `review_title_meta` for flagged rows, `no_action` otherwise.

Before coding the rule, two signal checks — one bucket table each, `n` printed — because the rule
leans on both:
1. **CTR-vs-position** (flag-linked: this is the signal behind FlyRank's CTR-fix logic) — is CTR
   really tier-dependent, or would a flat threshold do?
2. **Volume** (flag-linked: the signal behind quick-win logic) — does keyword `search_volume`
   actually predict a bigger opportunity gap, the way "go after high-volume keywords first" logic
   assumes?

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import json, os

REPO = "hf://datasets/FlyRank/internship-warehouse"
os.makedirs("work/outputs", exist_ok=True)

fact = pd.read_parquet(f"{REPO}/fact_content_daily_performance/month=2026-03/data_0.parquet")
dim_content = pd.read_parquet(f"{REPO}/dim_content.parquet")

fact_avail = fact[fact["gsc_data_available"] == True].copy()
agg = (fact_avail.groupby(["client_hash_id", "content_hash_id"])
       .agg(impressions=("gsc_impressions", "sum"),
            clicks=("gsc_clicks", "sum"),
            sum_position=("gsc_sum_position", "sum"))
       .reset_index())
agg["avg_position"] = agg["sum_position"] / agg["impressions"]
agg["ctr"] = agg["clicks"] / agg["impressions"] * 100

VISIBLE_MIN_IMPR = 150
visible = agg[(agg["impressions"] >= VISIBLE_MIN_IMPR) & (agg["avg_position"] > 0)].copy()

bins, labels = [0, 3, 10, 20, 50, np.inf], ["top_3", "page_1", "striking", "page_3_5", "deep"]
visible["position_tier"] = pd.cut(visible["avg_position"], bins=bins, labels=labels)
visible["tier_median_ctr"] = visible.groupby("position_tier", observed=True)["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - visible["tier_median_ctr"]

content_cols = ["client_hash_id", "content_hash_id", "content_type", "main_intent", "word_count", "search_volume"]
visible = visible.merge(dim_content[content_cols], on=["client_hash_id", "content_hash_id"], how="left")

print("candidate pool:", visible.shape)

# ===== SIGNAL 1: CTR-vs-position (flag-linked: CTR-fix logic) =====
sig1 = visible.groupby("position_tier", observed=True)["ctr"].agg(median_ctr="median", n="count")
print("\n=== SIGNAL 1: CTR by position tier ===")
print(sig1)

# ===== SIGNAL 2: volume (flag-linked: quick-win logic) =====
visible["volume_bucket"] = pd.cut(visible["search_volume"], bins=[-1, 0, 100, 1000, 1e9],
                                   labels=["zero", "1-100", "100-1000", "1000+"])
sig2 = visible.groupby("volume_bucket", observed=True)["ctr_gap"].agg(mean_gap="mean", median_gap="median", n="count")
print("\n=== SIGNAL 2: ctr_gap by search_volume bucket ===")
print(sig2)

candidate pool: (91974, 14)

=== SIGNAL 1: CTR by position tier ===
               median_ctr      n
position_tier                   
top_3            0.220022   9733
page_1           0.201265  44463
striking         0.124069  17631
page_3_5         0.031636  17303
deep             0.000000   2844

=== SIGNAL 2: ctr_gap by search_volume bucket ===
               mean_gap  median_gap      n
volume_bucket                             
zero           0.129727    0.007224  34459
1-100          0.114192    0.000000  46364
100-1000       0.044495   -0.031636   8111
1000+          0.013431   -0.031636   1558


**Verdicts:**

- **Signal 1 — CTR-vs-position: `CONFIRMED`.** Median CTR falls monotonically as position gets
  worse — top_3 0.220 → page_1 0.201 → striking 0.124 → page_3_5 0.032 → deep 0.000 — across five
  well-powered buckets (n from 2,844 to 44,463). Tier-adjusting CTR is doing real work, not
  chasing noise. This is the load-bearing assumption behind the whole rule.

- **Signal 2 — volume: `OPPOSITE`.** The naive "higher search volume = bigger opportunity" assumption
  behind quick-win logic runs backwards here: mean `ctr_gap` *shrinks* as volume rises (0.130 at
  zero-volume → 0.114 → 0.044 → 0.013 at 1000+), and the median gap actually goes *negative*
  (-0.032) for the two higher-volume buckets while sitting at ~0 for lower volume. In this slice,
  the pages most below their tier peers cluster in the low/zero-volume group, not the high-volume
  one. That's a clearly negative result, and a useful one — it's why the rule below scores by
  observed `impressions` (actual traffic already happening) instead of `search_volume` (theoretical
  keyword demand). Using `search_volume` to prioritize would have pointed the queue at the wrong
  pages.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = how far below tier-median CTR, times actual impressions — zero for pages at or above their
tier's typical CTR. Readable on purpose, no fitted weights.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

visible["underperforming"] = (visible["ctr_gap"] < 0).astype(int)
visible["score"] = visible["underperforming"] * (-visible["ctr_gap"]) * visible["impressions"]
visible["reason_code"] = np.where(visible["score"] > 0, "ctr_below_tier_peers", "at_or_above_tier_peers")
visible["action"] = np.where(visible["score"] > 0, "review_title_meta", "no_action")

queue = visible.sort_values("score", ascending=False).reset_index(drop=True)
out_cols = ["client_hash_id", "content_hash_id", "score", "reason_code", "action",
            "position_tier", "ctr", "tier_median_ctr", "ctr_gap", "impressions",
            "content_type", "main_intent", "word_count", "search_volume"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
print("flagged (score>0):", (queue["score"] > 0).sum(), "of", len(queue),
      f"({(queue['score'] > 0).mean():.1%})")

metrics = {
    "candidate_pool": int(len(visible)),
    "flagged_count": int((queue["score"] > 0).sum()),
    "flagged_rate": float((queue["score"] > 0).mean()),
    "signal1_ctr_by_tier": sig1.reset_index().to_dict(orient="records"),
    "signal2_gap_by_volume": sig2.reset_index().to_dict(orient="records"),
}
with open("work/outputs/w04_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print("wrote work/outputs/w04_metrics.json")

wrote 91974 rows to work/outputs/baseline_action_score.csv
flagged (score>0): 44560 of 91974 (48.4%)
wrote work/outputs/w04_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(queue[out_cols].head(10).to_string())

            client_hash_id           content_hash_id         score           reason_code             action position_tier       ctr  tier_median_ctr   ctr_gap  impressions     content_type    main_intent  word_count  search_volume
0  client_23a62021009f63c4  content_44f34c0a90047651  44333.553355  ctr_below_tier_peers  review_title_meta         top_3  0.011299         0.220022 -0.208723       212404  keyword article     commercial      3495.0            0.0
1  client_73cda7b4e4f265ea  content_8e1334d6356668e3  29599.449945  ctr_below_tier_peers  review_title_meta         top_3  0.000741         0.220022 -0.219281       134984  keyword article     commercial         NaN          720.0
2  client_73cda7b4e4f265ea  content_fec55986a1868d62  27199.229923  ctr_below_tier_peers  review_title_meta         top_3  0.000806         0.220022 -0.219216       124075  keyword article  informational         NaN          110.0
3  client_62f4a7e64f5e0096  content_34a70fea29d15f24  24484.732605  ctr_belo

1. `44f34c0a90047651` — **review_title_meta.** top_3 position, CTR 1.1% vs tier's 22.0%, 212k
   impressions — huge, credible gap. *Wrong if:* `search_volume` shows 0 despite 212k real
   impressions, which is itself a data-quality flag worth checking before trusting anything else
   about this row.
2. `8e1334d6356668e3` — **review_title_meta.** top_3, CTR 0.07% (near-zero) on 135k impressions.
   *Wrong if:* a CTR this close to zero at a top position, at this volume, is a tracking/attribution
   glitch rather than a real title problem — worth a sanity check before assuming it's fixable by
   copy.
3. `fec55986a1868d62` — **review_title_meta.** Same client as #2, same near-zero-CTR pattern (0.08%
   at top_3, 124k impressions). *Wrong if:* two near-identical anomalies from one client suggests a
   client-level tracking issue, not two independent title problems.
4. `34a70fea29d15f24` — **review_title_meta.** page_1, CTR 3.0% vs tier's 20.1%, 143k impressions,
   word_count and search_volume both present and plausible. *Wrong if:* nothing obvious — this is
   the kind of row the rule is built for.
5. `f6116743b00afc2d` — **review_title_meta.** page_1, CTR 1.4% vs 20.1%, 108k impressions.
   *Wrong if:* commercial intent pages this far below peers sometimes reflect a pricing/CTA mismatch
   rather than title wording — worth distinguishing before prescribing a title-only fix.
6. `7c6373141eae744a` — **review_title_meta.** page_1, CTR 6.3% vs 20.1%, missing `word_count`.
   *Wrong if:* can't judge whether this is a thin-content problem instead of a title problem without
   knowing word count first.
7. `9c057b66c30a3abb` — **review_title_meta.** Third row from `client_73cda7b4e4f265ea` in the top
   10, again near-zero CTR (0.12%) and missing `word_count`. *Wrong if:* by now this client alone
   accounts for 3 of the top 10 — the rule may just be surfacing one client's tracking anomaly
   three times, not three real content problems (see Section 4).
8. `cd3d932d4e1c8db0` — **review_title_meta.** page_1, CTR 0.4% vs 20.1%, and `search_volume` is
   missing entirely (not zero — actually absent). *Wrong if:* an editor can't judge keyword demand
   at all for this one; the review would proceed blind on that dimension.
9. `046fc480045b88f5` — **review_title_meta.** page_1, CTR 0.7% vs 20.1%, 84k impressions, all
   fields present and plausible. *Wrong if:* nothing stands out — a clean candidate.
10. `8d7d99f109e19aa2` — **review_title_meta.** top_3, CTR 14.2% vs 22.0% — the smallest gap in the
   top 10, only there because 203k impressions inflate the score. *Wrong if:* a 7.8-point gap at an
   already-decent 14% CTR may not be worth an editor's time compared to the rows above it — this is
   the weakest pick in the list (see Section 4).

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest pick: row 10 (`8d7d99f109e19aa2`).** Its CTR (14.2%) is already close to a healthy number
for its tier — the 7.8-point gap only lands it in the top 10 because 203k impressions dominate the
score. The rule conflates "biggest absolute opportunity" with "biggest score," and pure volume can
buy a spot even with a modest gap. A version 2 of this rule should probably rank by something closer
to relative gap (e.g. `ctr_gap / tier_median_ctr`) or cap the volume multiplier, so a huge-traffic
page with a small gap doesn't outrank a smaller-traffic page with a much larger one.

**A second weak pattern: client concentration.** `client_73cda7b4e4f265ea` alone supplies 3 of the
top 10 (rows 2, 3, 7), all sharing the same near-zero-CTR-at-top-position signature and missing
`word_count`. That's more consistent with one client-level tracking or feed issue than three
independent title problems — an editor working this list as-is would spend a third of their top-10
budget on what might be a single root cause. Worth a per-client cap or a "how many times has this
client already appeared" check in a later version.

**A third observation:** 4 of the top 10 rows have `word_count` missing (rows 2, 3, 6, 7) and one
has `search_volume` missing entirely (row 8). The rule doesn't currently require complete data to
flag a row — it should probably down-weight or hold out rows missing core content fields, since
`review_title_meta` presumes there's known content to review.

**Leakage check:** the score uses only `ctr_gap` (built from March 2026 GSC clicks/impressions,
already-observed) and `impressions` (same window) — no product flags (`health_score` and friends
don't exist in this dataset at all) and no future window (everything is March 2026, the same month
used to build the target in Week 3). Nothing here reaches past the decision moment.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

SCORE_INPUTS = ["ctr_gap", "impressions"]  # both built from March 2026 GSC data only
FUTURE_WINDOW_COLUMNS_USED = []             # none
PRODUCT_FLAG_COLUMNS_USED = []              # none exist in this dataset

print("score inputs:", SCORE_INPUTS)
print("future-window columns used:", FUTURE_WINDOW_COLUMNS_USED)
print("product-flag columns used:", PRODUCT_FLAG_COLUMNS_USED)

score inputs: ['ctr_gap', 'impressions']
future-window columns used: []
product-flag columns used: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.